In [1]:
# import qiskit libraries
from qiskit import Aer, QuantumCircuit, transpile, assemble, execute
from qiskit.visualization import plot_histogram
from qiskit.visualization import circuit_drawer

def oracle(list_number):
    """
    Creates an oracle that flips the sign of states corresponding to negative numbers.
    """
    n = len(list_number)  # Get the length of the list
    qc = QuantumCircuit(n+1)  # Create a quantum circuit
    
    # Flip the qubits that correspond to negative numbers
    for i, num in enumerate(list_number):
        if num < 0:
            qc.x(i)
    
    # Multi-controlled-Toffoli gate to mark the state if all conditions are satisfied
    qc.mct(list(range(n)), n)
    
    # Uncompute the flipping for qubits corresponding to negative numbers
    for i, num in enumerate(list_number):
        if num < 0:
            qc.x(i)
    
    return qc  # Return the oracle quantum circuit

def grover_diffusion(n):
    """
    Grover diffusion operator for n qubits.
    """
    qc = QuantumCircuit(n)  # Create a quantum circuit
    
    # Apply Hadamard gates on all qubits
    for qubit in range(n):
        qc.h(qubit)
        
    # Apply X (NOT) gates on all qubits
    for qubit in range(n):
        qc.x(qubit)
    
    # Apply controlled-Z gate
    qc.h(n-1)
    qc.mct(list(range(n-1)), n-1)
    qc.h(n-1)
    
    # Apply X (NOT) gates on all qubits to revert them back
    for qubit in range(n):
        qc.x(qubit)
        
    # Apply Hadamard gates on all qubits again
    for qubit in range(n):
        qc.h(qubit)
    
    return qc  # Return the diffusion quantum circuit

def find_negative_numbers(list_number):
    n = len(list_number)  # Get the length of the list
    qc = QuantumCircuit(n+1, n)  # Create a quantum circuit with n+1 qubits and n classical bits
    
    # Initialize qubits for Grover's algorithm
    qc.h(range(n))
    qc.x(n)
    qc.h(n)

    oracle_qc = oracle(list_number)  # Get the oracle quantum circuit
    diffusion_qc = grover_diffusion(n)  # Get the diffusion quantum circuit
    
    # Apply Grover's iteration (oracle + diffusion) sqrt(n) times
    for _ in range(int(n**0.5)):
        qc.append(oracle_qc, range(n+1))
        qc.append(diffusion_qc, range(n))

    # Measure the qubits
    qc.measure(range(n), range(n))
    
    # Simulate the quantum circuit
    aer_sim = Aer.get_backend('aer_simulator')
    t_qc = transpile(qc, aer_sim)
    qobj = assemble(t_qc)
    result = aer_sim.run(qobj).result()
    counts = result.get_counts(qc)
 
    # If the most common result is not all zeroes, then there's no negative number
    most_common = max(counts, key=counts.get)
    return '0' in most_common

# Test cases
A = find_negative_numbers([1, -3, 2, 15])
print(A)

B = find_negative_numbers([1, 4, 8, 11])
print(B)

C = find_negative_numbers([-15, -14, 2, -1])
print(C)

D = find_negative_numbers([0, 0, 0, 0])
print(D)

E = find_negative_numbers([0, 1, -2, 3])
print(E)

F = find_negative_numbers([1, 5, 234234, 123057])
print(F)


True
False
True
False
True
False


C:\Users\ameli\AppData\Local\Temp\ipykernel_28212\1573838448.py:82: DeprecationWarning: Using a qobj for run() is deprecated as of qiskit-aer 0.9.0 and will be removed no sooner than 3 months from that release date. Transpiled circuits should now be passed directly using `backend.run(circuits, **run_options).
  result = aer_sim.run(qobj).result()
